# Experiment 003: Volume Pairlist Layers — IchiV3 Comparison

Side-by-side comparison of all 6 arms testing incremental pairlist filtering layers:

| Arm | Description |
|-----|-------------|
| **A** | Baseline — StaticPairList, 107 pairs, no filtering |
| **B** | Binance volume pairlist — top 75 by volume |
| **B2** | Binance volume pairlist — top 100 by volume |
| **C** | Volume + GMX liquidity filter (min $500k) |
| **D** | Volume + liquidity + OI whale cap (2.5% of OI) |
| **E** | Volume + OI whale cap (no liquidity filter) |

**Strategy:** IchiV3_LS_Static (arms A-C, B2) / IchiV3_LS_Static_WhaleCap (arms D, E) | **Capital:** $100,000 | **Timerange:** 2021-01-06 to 2026-03-12

In [ ]:
import os, json, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import nest_asyncio

nest_asyncio.apply()
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('/home/ubuntu/dev/gmx-ccxt-freqtrade')
os.chdir(PROJECT_ROOT)

STARTING_BALANCE = 100_000
TEMPLATE = 'plotly_dark'

ARM_COLORS = ['#636EFA', '#00CC96', '#AB63FA', '#FFA15A', '#EF553B', '#19D3F3']
ARM_LABELS = ['Arm A: Baseline', 'Arm B: Volume 75', 'Arm B2: Volume 100', 'Arm C: Vol + Liq', 'Arm D: Vol + Liq + Whale', 'Arm E: Vol + Whale']
ARM_KEYS = ['arm_a', 'arm_b', 'arm_b2', 'arm_c', 'arm_d', 'arm_e']

RESULTS_DIR = PROJECT_ROOT / 'experiments/ichiv3-gmx/003-volume-pairlist-layers/results'
ZIP_PATHS = {
    'arm_a': RESULTS_DIR / 'arm_a_baseline.zip',
    'arm_b': RESULTS_DIR / 'arm_b_volume.zip',
    'arm_b2': RESULTS_DIR / 'arm_b2_volume_100.zip',
    'arm_c': RESULTS_DIR / 'arm_c_volume_liq.zip',
    'arm_d': RESULTS_DIR / 'arm_d_volume_liq_whale.zip',
    'arm_e': RESULTS_DIR / 'arm_e_volume_whale.zip',
}

for k, p in ZIP_PATHS.items():
    print(f'{k}: {p.name} — exists={p.exists()}')

In [ ]:
# --- Load all 6 result sets ---

def load_trades_from_zip(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        json_name = [n for n in z.namelist() if n.endswith('.json') and 'config' not in n and 'meta' not in n][0]
        with z.open(json_name) as f:
            data = json.load(f)
    strategy_name = list(data['strategy'].keys())[0]
    trades_list = data['strategy'][strategy_name]['trades']
    trades = pd.DataFrame(trades_list)
    trades['open_date'] = pd.to_datetime(trades['open_date'])
    trades['close_date'] = pd.to_datetime(trades['close_date'])
    return trades, strategy_name

all_trades = {}
for key, path in ZIP_PATHS.items():
    trades, strategy = load_trades_from_zip(path)
    all_trades[key] = trades
    print(f'{key}: {len(trades)} trades loaded ({strategy})')

In [ ]:
# --- Side-by-Side Metrics Table ---

def calculate_metrics_raw(trades, starting_balance=STARTING_BALANCE):
    """Return raw numeric metrics for comparison."""
    ts = trades.sort_values('close_date').copy()
    ts['cum_profit_abs'] = ts['profit_abs'].cumsum()
    ts['equity'] = starting_balance + ts['cum_profit_abs']
    
    total_profit_abs = ts['profit_abs'].sum()
    total_profit_pct = (total_profit_abs / starting_balance) * 100
    
    days = (ts['close_date'].max() - ts['open_date'].min()).days
    years = days / 365.25
    final_equity = starting_balance + total_profit_abs
    cagr = ((final_equity / starting_balance) ** (1 / years) - 1) * 100 if years > 0 else 0
    
    equity = ts['equity']
    rolling_max = equity.cummax()
    drawdown = (equity - rolling_max) / rolling_max * 100
    max_dd = drawdown.min()
    
    ts['close_day'] = ts['close_date'].dt.date
    daily_pnl = ts.groupby('close_day')['profit_abs'].sum()
    daily_returns = daily_pnl / starting_balance
    
    sharpe = (daily_returns.mean() / daily_returns.std()) * np.sqrt(365) if daily_returns.std() > 0 else 0
    downside = daily_returns[daily_returns < 0]
    sortino = (daily_returns.mean() / downside.std()) * np.sqrt(365) if len(downside) > 0 and downside.std() > 0 else 0
    calmar = abs(cagr / max_dd) if max_dd != 0 else 0
    
    win_rate = (ts['profit_abs'] > 0).mean() * 100
    
    gross_profit = ts.loc[ts['profit_abs'] > 0, 'profit_abs'].sum()
    gross_loss = abs(ts.loc[ts['profit_abs'] < 0, 'profit_abs'].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
    
    # Long/short breakdown
    if 'is_short' in ts.columns:
        longs = ts[~ts['is_short']]
        shorts = ts[ts['is_short']]
    else:
        longs = ts[ts['trade_direction'] == 'long']
        shorts = ts[ts['trade_direction'] == 'short']
    
    return {
        'Trades': len(ts),
        'Longs / Shorts': f"{len(longs)} / {len(shorts)}",
        'Profit ($)': round(total_profit_abs, 0),
        'Profit (%)': round(total_profit_pct, 1),
        'CAGR (%)': round(cagr, 1),
        'Max DD (%)': round(max_dd, 1),
        'Sharpe': round(sharpe, 2),
        'Sortino': round(sortino, 2),
        'Calmar': round(calmar, 2),
        'Win Rate (%)': round(win_rate, 1),
        'Profit Factor': round(profit_factor, 2),
        'Avg Trade (%)': round(ts['profit_ratio'].mean() * 100, 2),
        'Avg Stake ($)': round(ts['stake_amount'].mean(), 0),
        'Long P&L ($)': round(longs['profit_abs'].sum(), 0),
        'Short P&L ($)': round(shorts['profit_abs'].sum(), 0),
    }

metrics_all = {}
for key, label in zip(ARM_KEYS, ARM_LABELS):
    metrics_all[label] = calculate_metrics_raw(all_trades[key])

metrics_df = pd.DataFrame(metrics_all)
metrics_df

In [ ]:
# --- Delta Table vs Arm A Baseline ---
# Only compute deltas for numeric rows

baseline_col = ARM_LABELS[0]
numeric_metrics = metrics_df.loc[metrics_df[baseline_col].apply(lambda x: isinstance(x, (int, float)))]

delta_df = numeric_metrics.copy()
for col in delta_df.columns:
    if col != baseline_col:
        delta_df[col] = delta_df[col] - delta_df[baseline_col]

delta_df[baseline_col] = '(baseline)'

print('Delta vs Arm A Baseline (positive = higher than baseline):')
delta_df

In [ ]:
# --- Overlaid Equity Curves ---

fig = go.Figure()

for key, label, color in zip(ARM_KEYS, ARM_LABELS, ARM_COLORS):
    ts = all_trades[key].sort_values('close_date').copy()
    ts['cum_profit_abs'] = ts['profit_abs'].cumsum()
    ts['equity'] = STARTING_BALANCE + ts['cum_profit_abs']
    fig.add_trace(go.Scatter(
        x=ts['close_date'],
        y=ts['equity'],
        mode='lines',
        name=label,
        line=dict(color=color, width=2),
    ))

fig.update_layout(
    title='Equity Curves — All Arms',
    xaxis_title='Date',
    yaxis_title='Equity ($)',
    template=TEMPLATE,
    height=600,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01),
)
fig.show()

In [ ]:
# --- Drawdown Comparison ---

fig = go.Figure()

for key, label, color in zip(ARM_KEYS, ARM_LABELS, ARM_COLORS):
    ts = all_trades[key].sort_values('close_date').copy()
    ts['cum_profit_abs'] = ts['profit_abs'].cumsum()
    ts['equity'] = STARTING_BALANCE + ts['cum_profit_abs']
    rolling_max = ts['equity'].cummax()
    ts['drawdown_pct'] = (ts['equity'] - rolling_max) / rolling_max * 100
    fig.add_trace(go.Scatter(
        x=ts['close_date'],
        y=ts['drawdown_pct'],
        mode='lines',
        name=label,
        line=dict(color=color, width=1.5),
    ))

fig.update_layout(
    title='Drawdown Comparison — All Arms',
    xaxis_title='Date',
    yaxis_title='Drawdown (%)',
    template=TEMPLATE,
    height=500,
    legend=dict(yanchor='bottom', y=0.01, xanchor='left', x=0.01),
)
fig.show()

In [ ]:
# --- Long vs Short Profit — Grouped by Arm ---

ls_data = []
for key, label in zip(ARM_KEYS, ARM_LABELS):
    t = all_trades[key].copy()
    if 'is_short' in t.columns:
        t['direction'] = t['is_short'].apply(lambda x: 'Short' if x else 'Long')
    else:
        t['direction'] = t['trade_direction'].apply(lambda x: 'Short' if 'short' in str(x).lower() else 'Long')
    for d in ['Long', 'Short']:
        profit = t.loc[t['direction'] == d, 'profit_abs'].sum()
        ls_data.append({'Arm': label, 'Direction': d, 'Profit': profit})

ls_df = pd.DataFrame(ls_data)

fig = go.Figure()
for i, d in enumerate(['Long', 'Short']):
    subset = ls_df[ls_df['Direction'] == d]
    fig.add_trace(go.Bar(
        x=subset['Arm'],
        y=subset['Profit'],
        name=d,
        marker_color='#636EFA' if d == 'Long' else '#EF553B',
        text=subset['Profit'].apply(lambda x: f'${x:,.0f}'),
        textposition='outside',
    ))

fig.update_layout(
    title='Long vs Short Profit — By Arm',
    xaxis_title='Arm',
    yaxis_title='Profit ($)',
    barmode='group',
    template=TEMPLATE,
    height=500,
)
fig.show()

In [ ]:
# --- Correlation Matrix (Daily Returns) ---

daily_returns_dict = {}
for key, label in zip(ARM_KEYS, ARM_LABELS):
    ts = all_trades[key].sort_values('close_date').copy()
    ts['close_day'] = ts['close_date'].dt.date
    daily_pnl = ts.groupby('close_day')['profit_abs'].sum()
    daily_returns_dict[label] = daily_pnl / STARTING_BALANCE

daily_ret_df = pd.DataFrame(daily_returns_dict).fillna(0)
corr = daily_ret_df.corr()

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.index.tolist(),
    colorscale='RdBu',
    zmid=0,
    text=np.round(corr.values, 3),
    texttemplate='%{text}',
    textfont=dict(size=14),
))
fig.update_layout(
    title='Daily Returns Correlation Matrix',
    template=TEMPLATE,
    height=500,
    width=700,
)
fig.show()

In [ ]:
# --- Monthly Returns Comparison (Heatmap) ---

monthly_data = {}
for key, label in zip(ARM_KEYS, ARM_LABELS):
    ts = all_trades[key].sort_values('close_date').copy()
    ts['month'] = ts['close_date'].dt.to_period('M')
    monthly_pnl = ts.groupby('month')['profit_abs'].sum()
    monthly_ret = (monthly_pnl / STARTING_BALANCE) * 100
    monthly_data[label] = monthly_ret

monthly_df = pd.DataFrame(monthly_data)
monthly_df.index = monthly_df.index.astype(str)
monthly_df = monthly_df.fillna(0)

fig = go.Figure(data=go.Heatmap(
    z=monthly_df.T.values,
    x=monthly_df.index.tolist(),
    y=monthly_df.columns.tolist(),
    colorscale='RdYlGn',
    zmid=0,
    text=np.round(monthly_df.T.values, 1),
    texttemplate='%{text}%',
    textfont=dict(size=8),
))
fig.update_layout(
    title='Monthly Returns (% of Starting Capital) — All Arms',
    xaxis_title='Month',
    template=TEMPLATE,
    height=400,
    xaxis_tickangle=-45,
)
fig.show()

In [ ]:
# --- Trade Count by Arm ---

trade_counts = [len(all_trades[k]) for k in ARM_KEYS]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=ARM_LABELS,
    y=trade_counts,
    marker_color=ARM_COLORS,
    text=trade_counts,
    textposition='outside',
))
fig.update_layout(
    title='Total Trade Count by Arm',
    xaxis_title='Arm',
    yaxis_title='Number of Trades',
    template=TEMPLATE,
    height=450,
)
fig.show()

## Summary Findings

**Experiment 003 — Volume Pairlist Layers (IchiV3 with Historical Market Cap Sizing)**

Strategy uses point-in-time market cap data (2021-01-01 to 2026-03-13, 105 symbols from CoinGecko) for position sizing. Market cap tier multipliers from `IchiV3_LS_Static.json`: Blue chip (>$100B) = 1.0x, Large cap (>$10B) = 2.0x, Mid cap (>$1B) = 2.0x, Degen (<$1B) = 1.5x. Per-pair cap = 20% of balance.

**Whale cap (Arms D & E):** `max_oi_share = 0.025` — position capped at 2.5% of pair's total GMX open interest. Applied after market cap sizing. Verified: 0 violations across 638 OI-matched trades.

**Key observations:**

1. **Arm A (Baseline, 107 pairs)** — highest absolute profit ($397k, 41.7% CAGR) but also highest max DD (-37.1%). Sharpe 1.37.

2. **Arm D (Vol + Liq + WhaleCap)** — best Sharpe (1.48) and strong Sortino (3.22). Whale cap concentrates into fewer, larger trades (avg stake $43k vs $30k baseline). $212k profit with -29.5% DD.

3. **Volume filtering hurts more than it helps** — Arms B/B2 reduce profit without meaningfully improving risk-adjusted returns. Top-100 (B2) outperforms top-75 (B), suggesting the extra pairs add alpha.

4. **Liquidity filter is very aggressive** — $500k threshold cuts trade count in half (2084 → 1021) and drops profit by 66%. Only ~14 tokens pass consistently.

5. **Arm E (Vol + WhaleCap, no liq)** — worst risk-adjusted performance (Sharpe 0.98, Calmar 0.52, DD -37.8%). Whale cap without liquidity filter spreads capital too thin across low-OI tokens.

6. **Shorts dominate** — 70%+ of trades are shorts, generating 80-90% of total P&L across all arms.

7. **vs IchiV2 (Exp 002):** IchiV3 outperforms on absolute profit across all arms due to market cap sizing. The D arm (Vol+Liq+WhaleCap) shows the largest improvement — market cap sizing + whale cap interact favorably.

In [ ]:
# --- Cross-Experiment Comparison: IchiV2 (Exp 002) vs IchiV3 (Exp 003) ---

EXP002_RESULTS_DIR = PROJECT_ROOT / 'experiments/ichiv2-gmx/002-volume-pairlist-layers/results'
EXP002_ZIP_PATHS = {
    'arm_a': EXP002_RESULTS_DIR / 'arm_a_baseline.zip',
    'arm_b': EXP002_RESULTS_DIR / 'arm_b_volume.zip',
    'arm_b2': EXP002_RESULTS_DIR / 'arm_b2_volume_100.zip',
    'arm_c': EXP002_RESULTS_DIR / 'arm_c_volume_liq.zip',
    'arm_d': EXP002_RESULTS_DIR / 'arm_d_volume_liq_whale.zip',
    'arm_e': EXP002_RESULTS_DIR / 'arm_e_volume_whale.zip',
}

# Load exp002 trades
exp002_trades = {}
for key, path in EXP002_ZIP_PATHS.items():
    if path.exists():
        trades, strategy = load_trades_from_zip(path)
        exp002_trades[key] = trades
        print(f'Exp002 {key}: {len(trades)} trades ({strategy})')
    else:
        print(f'Exp002 {key}: MISSING ({path.name})')

# Build side-by-side table focusing on key metrics
KEY_METRICS = ['Trades', 'Profit (%)', 'CAGR (%)', 'Max DD (%)', 'Sharpe', 'Sortino', 'Calmar', 'Win Rate (%)', 'Profit Factor']

comparison_rows = []
for key, label in zip(ARM_KEYS, ARM_LABELS):
    if key not in exp002_trades:
        continue
    v2_m = calculate_metrics_raw(exp002_trades[key])
    v3_m = calculate_metrics_raw(all_trades[key])
    row = {'Arm': label}
    for m in KEY_METRICS:
        v2_val = v2_m.get(m, 0)
        v3_val = v3_m.get(m, 0)
        if isinstance(v2_val, (int, float)) and isinstance(v3_val, (int, float)):
            row[f'V2 {m}'] = v2_val
            row[f'V3 {m}'] = v3_val
            row[f'Δ {m}'] = round(v3_val - v2_val, 2)
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).set_index('Arm')

# Reorder: group by metric
ordered_cols = []
for m in KEY_METRICS:
    for prefix in [f'V2 {m}', f'V3 {m}', f'Δ {m}']:
        if prefix in comparison_df.columns:
            ordered_cols.append(prefix)
comparison_df = comparison_df[ordered_cols]

print('IchiV2 (Exp 002) vs IchiV3 (Exp 003) — Cross-Experiment Comparison')
print('Δ = V3 - V2 (positive = V3 better for profit/Sharpe, negative = V3 better for DD)')
print()
comparison_df